# Sanity check: Common Voice Mongolian + Whisper (learning notebook)

This notebook walks through **data layout**, **audio**, **tokenizer / log-mel features**, a **base Whisper** smoke test, a **training collator** batch, and a **one-batch overfit** check — the same building blocks used by `src/` and `run_train.py`.

**Working directory:** open Jupyter from `whisper-medium-mn-lora` (this folder), or run the first code cell so `PROJECT_ROOT` points here and `import src` works.

**Training:** `python run_train.py` (see `--help`). **Demo:** `python gradio_demo.py`.

Reference copies only (not required for this notebook): `train_whisper_mn.py`, `training.ipynb`.

## 1. Project path and imports

In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath(".")
if not os.path.isfile(os.path.join(PROJECT_ROOT, "run_train.py")):
    PROJECT_ROOT = os.path.abspath(os.path.join(".", "whisper-medium-mn-lora"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
assert os.path.isdir(os.path.join(PROJECT_ROOT, "src")), "Expected src/ under PROJECT_ROOT"

PROJECT_ROOT: /home/toru2/Amara/Deep_learning/lab3/whisper-medium-mn-lora


In [2]:
import numpy as np
import pandas as pd
import soundfile as sf
import torch

print("PyTorch:", torch.__version__)
print("CUDA: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:    ", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA:  True
GPU:     NVIDIA GeForce RTX 5080


## 2. Find Common Voice folder

Expected layout: `validated.tsv` and `clips/` with audio files referenced in the TSV.

In [3]:
DATA_ROOT = None
for rel in ("../common_voice_mn", "common_voice_mn", "../../lab3/common_voice_mn"):
    cand = os.path.abspath(os.path.join(PROJECT_ROOT, rel))
    if os.path.isfile(os.path.join(cand, "validated.tsv")) and os.path.isdir(
        os.path.join(cand, "clips")
    ):
        DATA_ROOT = cand
        break

if DATA_ROOT is None:
    raise FileNotFoundError("Set DATA_ROOT manually to your common_voice_mn directory")

print("DATA_ROOT:", DATA_ROOT)

DATA_ROOT: /home/toru2/Amara/Deep_learning/lab3/common_voice_mn


## 3. Peek at `validated.tsv` (pandas)

We only need paths and sentences for this pipeline; other columns exist in the full Mozilla export.

In [4]:
from IPython.display import display

tsv = os.path.join(DATA_ROOT, "validated.tsv")
df = pd.read_csv(tsv, sep="\t")
print("rows:", len(df))
display(df.head(3))
df["sentence_len"] = df["sentence"].str.len()
print("sentence char length — min/median/max:")
print(df["sentence_len"].min(), df["sentence_len"].median(), df["sentence_len"].max())

rows: 33361


,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
0,0af16655992bdd279a35058cea3469a1c6b114d0981d44...,common_voice_mn_37556246.mp3,2f1db8e6e491f0aa2b5f18084e18ed702d4399d7eceaf7...,Цагаан соёотын дүнсгэр араншин бол олон жилийн...,NaN,2,1,NaN,NaN,NaN,NaN,mn,NaN
1,119ca982a55f636c86143e4c1a8a729a04da66751dbcac...,common_voice_mn_18785480.mp3,d2abf5a2b246d332322ed838f8d5747b6b862326e5b542...,"Иймэрхүү хоол идсээр байгаад дэндүү туранхай, ...",NaN,2,0,twenties,female_feminine,NaN,NaN,mn,NaN
2,146548cc5ad2f6f60debd2c7a1ba98935b10842580862a...,common_voice_mn_22947513.mp3,f041cbef9fbe938c4068c73d47d8d7d94442a030c45cf2...,Хорт хавдраар өвчлөх явдлыг эрс бууруулахаар э...,NaN,2,1,twenties,male_masculine,NaN,NaN,mn,NaN


sentence char length — min/median/max:
6 62.0 16984


## 4. Audio duration (soundfile headers, no full decode)

Training filters by duration; sampling rows is enough to see the distribution.

In [5]:
clips = os.path.join(DATA_ROOT, "clips")
n_sample = min(500, len(df))
durations = []
for _, row in df.head(n_sample).iterrows():
    path = os.path.join(clips, row["path"])
    if not os.path.isfile(path):
        continue
    try:
        durations.append(sf.info(path).duration)
    except Exception:
        pass
durations = np.array(durations)
print(f"sampled files: {len(durations)}")
print("min / mean / max sec:", durations.min(), durations.mean(), durations.max())

sampled files: 500
min / mean / max sec: 2.0235 5.861871312499999 10.5555


## 5. Hugging Face auth (optional)

Only needed if you download private models or will push later. Set `HF_TOKEN` in the environment or a `.env` in `PROJECT_ROOT`.

In [6]:
from huggingface_hub import login

import os
from pathlib import Path

token = os.environ.get("HF_TOKEN")
if not token and Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        line = line.strip()
        if line.startswith("HF_TOKEN="):
            token = line.split("=", 1)[1].strip()
            break
if token:
    login(token=token)
    print("[OK] Logged in to Hugging Face Hub")
else:
    print("[SKIP] No HF_TOKEN — public model download should still work")

[SKIP] No HF_TOKEN — public model download should still work


## 6. Same pipeline as training: `src.config` + `src.data`

Constants (`MODEL_NAME`, duration limits, `MAX_TARGET_TOKENS`) live in one place so the notebook matches `run_train.py`.

In [7]:
from transformers import WhisperTokenizer, WhisperProcessor, WhisperFeatureExtractor

from src.config import LANGUAGE, MODEL_NAME, TASK, MAX_AUDIO_SEC, MIN_AUDIO_SEC
from src.data import load_common_voice_validated, filter_dataset, split_dataset

print("MODEL_NAME:", MODEL_NAME)
print("LANGUAGE / TASK:", LANGUAGE, TASK)
print("duration window (s):", MIN_AUDIO_SEC, "..", MAX_AUDIO_SEC)

tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

ds = load_common_voice_validated(DATA_ROOT)
ds = filter_dataset(ds, tokenizer)
dsd = split_dataset(ds)

MODEL_NAME: openai/whisper-medium
LANGUAGE / TASK: Mongolian transcribe
duration window (s): 1.0 .. 30.0
[INFO] Loaded validated.tsv: 33361 rows


Filter:   0%|          | 0/33361 [00:00<?, ? examples/s]

[INFO] Filtered: 33361 -> 33343 samples (removed 18 by duration/label length)
         train: 26674 samples
    validation: 3334 samples
          test: 3335 samples


## 7. Listen to one clip (optional)

Requires `ipywidgets` in some environments.

In [8]:
from IPython.display import Audio, display

sample = dsd["test"][0]
print("REF:", sample["sentence"])
print("PATH:", sample["audio_path"])
audio, sr = sf.read(sample["audio_path"], dtype="float32")
if audio.ndim > 1:
    audio = audio.mean(axis=1)
display(Audio(audio, rate=sr))

REF: Мөн хэлийг ойлгох, сурахад зориулсан загварууд, тэмдгийн ба магадлалын загварын харьцуулал бий болжээ.
PATH: /home/toru2/Amara/Deep_learning/lab3/common_voice_mn/clips/common_voice_mn_20541879.mp3


## 8. Log-mel shape for one utterance

Whisper expects 16 kHz mono; `load_audio_array` in `src.collate` matches training batches.

In [9]:
from src.collate import load_audio_array

arr = load_audio_array(sample["audio_path"])
inputs = feature_extractor(arr, sampling_rate=16_000, return_tensors="pt")
print("input_features shape:", tuple(inputs.input_features.shape))

input_features shape: (1, 80, 3000)


## 9. Base Whisper: generate on a few test lines

Loads ~1.5B params — use GPU if available. This is **before** LoRA fine-tuning.

In [10]:
from transformers import WhisperForConditionalGeneration
import evaluate

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)

metric = evaluate.load("wer")
preds, refs = [], []
k = min(3, len(dsd["test"]))
for i in range(k):
    ex = dsd["test"][i]
    arr = load_audio_array(ex["audio_path"])
    inp = feature_extractor(arr, sampling_rate=16_000, return_tensors="pt")
    feats = inp.input_features.to(model.device, dtype=torch.bfloat16)
    with torch.no_grad():
        ids = model.generate(input_features=feats)
    hyp = tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    preds.append(hyp)
    refs.append(ex["sentence"])
    print(f"--- sample {i} ---")
    print("REF:", ex["sentence"])
    print("HYP:", hyp)

print("WER (%):", 100 * metric.compute(predictions=preds, references=refs))

del model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

--- sample 0 ---
REF: Мөн хэлийг ойлгох, сурахад зориулсан загварууд, тэмдгийн ба магадлалын загварын харьцуулал бий болжээ.
HYP:  The first layer of the moon is the layer that is the most important.
--- sample 1 ---
REF: Түүнд Холбоотныг нэгтгэх сэтгэлийн дотоод тэнхээ байгаагүй.
HYP:  The next day, the children were born.
--- sample 2 ---
REF: Бүр үнээ бүрээсээ найман зуун литр гэсэн байх аа?
HYP:  I'm going to make a little bit of a cake.
WER (%): 106.89655172413792


## 10. Training collator: one mini-batch

Same `DataCollatorSpeechSeq2SeqWithPadding` as `src.train.run_training`. Checks tensor dtypes and shapes before a long run.

In [11]:
from src.collate import DataCollatorSpeechSeq2SeqWithPadding

collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
rows = [dsd["train"][i] for i in range(2)]
batch = collator(rows)
print("keys:", batch.keys())
print("input_features:", batch["input_features"].shape, batch["input_features"].dtype)
print("labels:", batch["labels"].shape, batch["labels"].dtype)

keys: KeysView({'input_features': tensor([[[-0.7695, -0.7695, -0.7695,  ..., -0.7695, -0.7695, -0.7695],
         [-0.7695, -0.7695, -0.7695,  ..., -0.7695, -0.7695, -0.7695],
         [-0.7695, -0.7695, -0.7695,  ..., -0.7695, -0.7695, -0.7695],
         ...,
         [-0.7695, -0.7695, -0.7695,  ..., -0.7695, -0.7695, -0.7695],
         [-0.7695, -0.7695, -0.7695,  ..., -0.7695, -0.7695, -0.7695],
         [-0.7695, -0.7695, -0.7695,  ..., -0.7695, -0.7695, -0.7695]],

        [[-0.6602, -0.6602, -0.6602,  ..., -0.6602, -0.6602, -0.6602],
         [-0.6602, -0.6602, -0.6602,  ..., -0.6602, -0.6602, -0.6602],
         [-0.6602, -0.6602, -0.6602,  ..., -0.6602, -0.6602, -0.6602],
         ...,
         [-0.6602, -0.6602, -0.6602,  ..., -0.6602, -0.6602, -0.6602],
         [-0.6602, -0.6602, -0.6602,  ..., -0.6602, -0.6602, -0.6602],
         [-0.6602, -0.6602, -0.6602,  ..., -0.6602, -0.6602, -0.6602]]],
       dtype=torch.bfloat16), 'labels': tensor([[50258, 50314, 50359, 50363, 18172

## 11. One-batch overfit (training stack sanity check)

**Goal:** Train LoRA on a **single mini-batch** (2 examples) for several steps. **Training loss should fall** if the collator, model, and backward pass match real training.

**Expectations:** This is not good generalization — it only checks that the pipeline can memorize one batch. Use a GPU if possible (`whisper-medium` is heavy on CPU).

Run **after** sections 6 and 10 so `dsd`, `processor`, and `tokenizer` exist.

In [12]:
import gc
import shutil
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
)

from src.collate import DataCollatorSpeechSeq2SeqWithPadding
from src.config import (
    GENERATION_MAX_LENGTH,
    LANGUAGE,
    LORA_ALPHA,
    LORA_DROPOUT,
    LORA_R,
    LORA_TARGET_MODULES,
    MODEL_NAME,
    TASK,
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# One batch: 2 rows from train
BATCH = 2
tiny_ds = Dataset.from_dict({
    "audio_path": [dsd["train"][i]["audio_path"] for i in range(BATCH)],
    "sentence": [dsd["train"][i]["sentence"] for i in range(BATCH)],
})
print("Tiny dataset (1 batch):", len(tiny_ds), "examples")

out_dir = Path(PROJECT_ROOT) / "sanity-overfit-tmp"
if out_dir.exists():
    shutil.rmtree(out_dir)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

ob_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if use_bf16 else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    ob_model = ob_model.to(torch.float32)

ob_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)
ob_model.config.suppress_tokens = []

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
)
ob_model.enable_input_require_grads()
ob_model = get_peft_model(ob_model, lora_config)
ob_model.print_trainable_parameters()

collator_ob = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

targs = Seq2SeqTrainingArguments(
    output_dir=str(out_dir),
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=1,
    learning_rate=1e-3,
    max_steps=40,
    bf16=use_bf16,
    fp16=False,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=0,
    generation_max_length=GENERATION_MAX_LENGTH,
    predict_with_generate=False,
)

trainer_ob = Seq2SeqTrainer(
    model=ob_model,
    args=targs,
    train_dataset=tiny_ds,
    eval_dataset=None,
    data_collator=collator_ob,
    processing_class=processor,
    compute_metrics=None,
)

result = trainer_ob.train()
print("train_runtime_s:", round(result.metrics.get("train_runtime", 0), 2))

losses = [
    h["loss"]
    for h in trainer_ob.state.log_history
    if "loss" in h
]
print("Loss each log step:", [round(x, 4) for x in losses])
if len(losses) >= 2 and losses[-1] < losses[0]:
    print("[OK] Loss decreased — training stack looks wired correctly.")
else:
    print("[WARN] Loss did not drop clearly; try more steps or check GPU/AMP.")

del trainer_ob, ob_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Tiny dataset (1 batch): 2 examples


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

trainable params: 4,718,592 || all params: 768,576,512 || trainable%: 0.6139


Step,Training Loss
5,3.331073
10,0.779345
15,0.715564
20,1.166844
25,1.271390
30,0.836892
35,0.604423
40,0.501413


train_runtime_s: 5.86
Loss each log step: [3.3311, 0.7793, 0.7156, 1.1668, 1.2714, 0.8369, 0.6044, 0.5014]
[OK] Loss decreased — training stack looks wired correctly.


## Next steps

- Train: `python run_train.py` (add `--eval` or `--push --hub-model-id USER/repo` as needed).
- TensorBoard: `tensorboard --logdir runs/`.
- After training: `python gradio_demo.py --adapter-path results/final-lora`.
- Optional: remove the overfit scratch dir `sanity-overfit-tmp/` if you do not need it.